# Exception Handling

Handle errors so your program fails gracefully, and recover when it can.

**You will learn to:**

- Read an error message
- Catch specific exceptions with `try` and `except`
- Use `else` and `finally`
- Raise your own exceptions and define custom ones
- Retry a flaky operation safely

## 1. What is an exception?

When Python cannot continue, it **raises an exception**. Unhandled, it stops the program and prints a traceback.
Read a traceback from the **bottom up**: the last line names the error, the lines above show where it happened.

| Exception | Typical cause |
|---|---|
| `ValueError` | Right type, wrong value: `int("abc")` |
| `TypeError` | Wrong type: `"a" + 1` |
| `KeyError` | Missing dictionary key |
| `IndexError` | List index out of range |
| `ZeroDivisionError` | Dividing by zero |
| `FileNotFoundError` | Opening a file that does not exist |
| `AttributeError` | Missing attribute or method |

## 2. `try` and `except`

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Cannot divide by zero")

print("The program keeps going")

Catch **specific** exceptions. Use `as` to see the message.

In [ ]:
def to_int(text):
    try:
        return int(text)
    except ValueError as error:
        print("Bad number:", error)
        return None


print(to_int("42"))
print(to_int("forty-two"))

In [ ]:
data = {"name": "Ada"}
values = [1, 2, 3]

for action in [lambda: data["age"], lambda: values[10], lambda: "a" + 1]:
    try:
        action()
    except (KeyError, IndexError) as error:       # one handler, several types
        print(f"Lookup problem: {type(error).__name__}: {error}")
    except TypeError as error:
        print(f"Type problem: {error}")

> **Avoid** a bare `except:` or `except Exception: pass`. It hides real bugs. Catch only what you can handle.

## 3. `else` and `finally`

- `else` runs when **no** exception happened.
- `finally` runs **always**, which makes it the place for clean-up.

In [ ]:
def read_number(text):
    try:
        number = int(text)
    except ValueError:
        print("Not a number")
    else:
        print("Parsed", number)
        return number
    finally:
        print("Done with", repr(text))


read_number("7")
read_number("x")

In [ ]:
# Files: "with" already closes for you, but finally is the general tool
try:
    with open("missing_file.txt", encoding="utf-8") as file:
        print(file.read())
except FileNotFoundError:
    print("File not found; using defaults")

## 4. Raise your own exceptions

In [ ]:
def set_temperature(value):
    if not 0 <= value <= 1:
        raise ValueError(f"temperature must be between 0 and 1, got {value}")
    return value


try:
    set_temperature(1.5)
except ValueError as error:
    print("Rejected:", error)

### Custom exceptions

In [ ]:
class PromptTooLongError(Exception):
    """Raised when a prompt is longer than the allowed limit."""


def check_prompt(prompt, max_chars=50):
    if len(prompt) > max_chars:
        raise PromptTooLongError(f"{len(prompt)} characters; limit is {max_chars}")
    return prompt


try:
    check_prompt("x" * 80)
except PromptTooLongError as error:
    print("Caught:", error)

### Chain exceptions

In [ ]:
class ConfigError(Exception):
    pass


def load_port(raw):
    try:
        return int(raw)
    except ValueError as error:
        raise ConfigError(f"Invalid port {raw!r}") from error      # keeps the original cause


try:
    load_port("eighty")
except ConfigError as error:
    print(error, "| caused by:", repr(error.__cause__))

## 5. Worked example: retry with back-off

Network and API calls fail for temporary reasons. Retry a few times, wait a little longer each time,
and give up with a clear error if it never works.

In [ ]:
import time


class TemporaryError(Exception):
    """A failure that may succeed if tried again."""


calls = {"count": 0}


def flaky_call():
    calls["count"] += 1
    if calls["count"] < 3:
        raise TemporaryError(f"service busy (call {calls['count']})")
    return "response text"


def with_retries(func, max_attempts=4, base_delay=0.05):
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except TemporaryError as error:
            print(f"Attempt {attempt} failed: {error}")
            if attempt == max_attempts:
                raise                                   # re-raise the last error
            time.sleep(base_delay * 2 ** (attempt - 1))  # 0.05s, 0.1s, 0.2s ...


print(with_retries(flaky_call))

Retry only errors that are **temporary**. Retrying a bad request, such as a wrong password, never helps.

## Try it

1. Write `safe_int(text, default=0)` that returns `default` when `text` is not a number.
2. Write `get_user_age(users, name)` that returns an age from a dictionary of users and raises a `KeyError` with a helpful message when the user is missing. Catch it in the caller.
3. Define `InvalidScoreError` and a function `grade(score)` that raises it for scores outside 0 to 100.

In [ ]:
# Your answers here